# Notebook 01 · Data Quality, Proxies, and Gap Characterization

This notebook executes **Phase A, Step 1** of the `heritageshm` pipeline:

1. **Sensor Loading** — Load the preprocessed sensor CSV from Notebook 00.
2. **Proxy Loading** — Explicitly load specific environmental proxies (e.g. Skin Temperature, Air Temp).
3. **Alignment** — Resample and synchronize multiple proxies onto the sensor index.
4. **Gap Characterization** — Classify missing data and diagnose gap taxonomy.
5. **Save** — Export the aligned, cleaned dataset to `/data/interim/aligned/`.

In [ ]:
import sys
import os
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

from heritageshm.dataloader import load_preprocessed_sensor, load_proxy_data, save_interim_data
from heritageshm.preprocessing import align_multiple_proxies
from heritageshm.diagnostics import characterize_gaps


## Step 1 · Load Preprocessed Sensor Data

In [ ]:
# Specify which station to analyze here
TARGET_STATION = 'st01'

SENSOR_FILE = f'data/interim/sensor/{TARGET_STATION}_preprocessed.csv'
df_sensor = load_preprocessed_sensor(SENSOR_FILE)

print(f"Loaded {TARGET_STATION} sensor dataset: {df_sensor.shape}")
df_sensor.head(3)

## Step 2 · Load Proxies Individually
Fetch any environmental proxy data you want to associate with the sensor.

In [ ]:
# Load specific proxy files into dataframes
# e.g., df_era5_skin = load_proxy_data('data/raw/proxies/era5_skin_temp.csv')

# Create a dictionary mapping a logical name to its dataframe
proxies_dict = {
    # 'skin_temp': df_era5_skin,
}
print(f"Loaded {len(proxies_dict)} proxy datasets.")

## Step 3 · Resampling and Alignment
Using `align_multiple_proxies`, we resample all defined proxies to match the desired monitoring frequency.

In [ ]:
TARGET_FREQ = '1H' # Hourly

if proxies_dict:
    df_aligned = align_multiple_proxies(df_sensor, proxies_dict, resample_freq=TARGET_FREQ)
else:
    print("No proxies loaded. Proceeding with sensor data only.")
    df_aligned = df_sensor.resample(TARGET_FREQ).mean()

df_aligned.head(3)

## Step 4 · Gap Characterization and Final Save
Diagnose the gaps, then save the unified dataset for analysis.

In [ ]:
try:
    gap_stats = characterize_gaps(df_aligned)
    print(gap_stats)
except Exception as e:
    print(f"Gap characterization skipped or failed: {e}")

OUTPUT_PATH = f'data/interim/aligned/{TARGET_STATION}_aligned_dataset.csv'
save_interim_data(df_aligned, OUTPUT_PATH)